# 第16章 利率期权（Cap/Floor/Swaption） — 编程实验完整解答

[![Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/albertandking/fixed-income/blob/main/notebooks/solutions/ch16_solutions.ipynb) [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/albertandking/fixed-income/main?labpath=notebooks/solutions/ch16_solutions.ipynb)

本 notebook 给出本章全部编程实验的完整可运行解答；联网（akshare）部分以注释/降级方式给出，离线也能跑通。


In [ ]:
# 自举单元：Colab/Binder 自动安装 fi；本地跳过。
import importlib.util, sys, subprocess
if importlib.util.find_spec('fi') is None:
    if 'google.colab' in sys.modules:
        subprocess.run(['git','clone','--depth','1','https://github.com/albertandking/fixed-income.git','/content/fi-book'],check=False)
        subprocess.run([sys.executable,'-m','pip','install','-e','/content/fi-book'],check=False)
    else:
        print('提示：仓库根目录执行 `uv sync --extra all` 后运行本 notebook。')


## 编程实验 7：例16.1-16.4 + 图16-1 + Cap-Floor 平价


In [ ]:
import numpy as np
from fi import rateopt as ro, plotting
plotting.use_chinese_style()
resets, taus = [1,2,3,4], [1,1,1,1]; pay_dfs = [1.03**-t for t in (2,3,4,5)]; fwds=[0.03]*4; N=1e8; vol=0.20
print('caplet=', round(ro.black_caplet(0.03,0.03,vol,1,1,pay_dfs[0],N,'cap'),0))
for K in (0.02,0.03,0.04):
    c=ro.black_cap(fwds,K,vol,resets,taus,pay_dfs,N,'cap'); f=ro.black_cap(fwds,K,vol,resets,taus,pay_dfs,N,'floor')
    swp=N*(sum(fwds[i]*taus[i]*pay_dfs[i] for i in range(4))-K*sum(pay_dfs))
    print(f'K={K*100:.0f}%: Cap-Floor={c-f:>12,.0f}  payerSwap={swp:>12,.0f}')
strikes = np.linspace(0.015,0.045,31)
fig, ax = plotting.new_axes()
ax.plot(strikes*100, [ro.black_cap(fwds,K,vol,resets,taus,pay_dfs,N,'cap')/1e4 for K in strikes], label='Cap')
ax.plot(strikes*100, [ro.black_cap(fwds,K,vol,resets,taus,pay_dfs,N,'floor')/1e4 for K in strikes], label='Floor')
ax.axvline(3, ls=':', color='gray'); ax.set_xlabel('执行价 K (%)'); ax.set_ylabel('价值(万元)'); ax.set_title('Cap/Floor 在 ATM 相交'); ax.legend(); fig.tight_layout()


## 编程实验 8：波动率微笑（合成价格反求隐含波动率）


In [ ]:
Ks = np.linspace(0.02,0.04,9)
true_vol = lambda K: 0.20 + 200*(K-0.03)**2   # 人造微笑
prices = [ro.black_caplet(0.03,K,true_vol(K),1,1,pay_dfs[0],N,'cap') for K in Ks]
ivs = [ro.implied_vol(p,0.03,K,1,1,pay_dfs[0],N,'cap') for p,K in zip(prices,Ks)]
fig, ax = plotting.new_axes()
ax.plot(Ks*100, np.array(ivs)*100, marker='o'); ax.set_xlabel('执行价 K (%)'); ax.set_ylabel('隐含波动率 (%)'); ax.set_title('波动率微笑'); fig.tight_layout()


## 编程实验 9：swaption 价格曲面（期权期限 × 执行价）


In [ ]:
expiries = [1,2,3,5]; strikes2 = [0.02,0.025,0.03,0.035,0.04]
ann = sum(pay_dfs)
print('payer swaption 价格(万元): 行=期权期限, 列=执行价')
print('期限\\K  ' + '  '.join(f'{K*100:.1f}%' for K in strikes2))
for T in expiries:
    row = [ro.black_swaption(0.03,K,vol,T,ann,N,'payer')/1e4 for K in strikes2]
    print(f'{T}y    ' + '  '.join(f'{x:6.1f}' for x in row))
print('对各价格除以年金、用 Black 反求即得隐含波动率曲面')
